# ProductBrand → Brand: useful evidence, or a cleaning rule?

We want to separate **missing labels** from **populated disagreements**. A frequent label is evidence, not ground truth. `UnItemised brand` and `Other Brands` stay valid values throughout this audit.

This notebook reads both original CSVs, learns relationships from training only, and produces review tables. It never fills, relabels, or writes datasets. Counts describe listings, not necessarily independent products. Read the cells in order; all full result tables stay available as named DataFrames.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data' / 'provided').is_dir():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'provided'
FILES = {name: DATA / f'Selfcare_{name}_data (1).csv' for name in ['Training', 'Target']}
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in FILES.items()}
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 85)
pd.set_option('display.max_rows', 60)

frames = []
for name, path in FILES.items():
    frame = pd.read_csv(path, dtype='string', keep_default_na=False, low_memory=False)
    frame['_dataset'] = name
    frame['_source_row'] = frame.index  # zero-based parsed CSV record, not physical line
    frames.append(frame)
data = pd.concat(frames, ignore_index=True)
source_columns = [c for c in data if not c.startswith('_')]
original_labels = data[['ProductBrand', 'Brand']].copy(deep=True)
BUCKETS = ['UnItemised brand', 'Other Brands']
CONTEXTS = ['Retailer', 'ProductCategory', 'Segment', 'Sub-Segment']

## 1. What counts as missing, and which rows are usable?

Only blank/whitespace and the literal `null` (case-insensitive) count as missing. We preserve literal `None`, `NA`, and other strings rather than guessing their meaning. Matching keys trim outer whitespace but preserve case, punctuation, and spelling. The original columns remain untouched.

Some records have shifted-looking fields. For a sensitivity check, use the same **provisional** screen as the identity audit: digit-only MDM ID, numeric timestamp between 40000 and 60000, and an HTTP(S) URL. This is not a validated schema or a deletion rule. We show full-export results first, then use the screened training view for candidate evidence. Failed-screen rows remain available and are never proposed for action.

In [2]:
def key(series):
    text = series.str.strip()
    return text.mask(text.eq('') | text.str.casefold().eq('null'))

for column in ['ProductBrand', 'Brand'] + CONTEXTS:
    data['_' + column] = key(data[column])
data['_screen'] = (
    data['MDM_Id'].str.strip().str.fullmatch(r'\d+', na=False)
    & pd.to_numeric(data['MDM_InsertDateTime'], errors='coerce').between(40000, 60000)
    & data['ProductUrl'].str.strip().str.match(r'^https?://', case=False, na=False)
).fillna(False)
train_all = data.loc[data['_dataset'].eq('Training')].copy()
target_all = data.loc[data['_dataset'].eq('Target')].copy()
train = train_all.loc[train_all['_screen']].copy()
target = target_all.loc[target_all['_screen']].copy()

scope = data.groupby('_dataset').agg(
    rows=('_source_row', 'size'), screened_rows=('_screen', 'sum'),
    missing_productbrand=('_ProductBrand', lambda s: s.isna().sum()),
    missing_brand=('_Brand', lambda s: s.isna().sum()),
)
scope['failed_screen'] = scope['rows'] - scope['screened_rows']
display(scope)
display(pd.DataFrame({
    'outer_whitespace_changes': [data[c].ne(data[c].str.strip()).sum() for c in ['ProductBrand', 'Brand']],
    'literal_None_kept': [data[c].eq('None').sum() for c in ['ProductBrand', 'Brand']],
}, index=['ProductBrand', 'Brand']))
failed_screen_rows = data.loc[~data['_screen']]
print('Failed-screen records remain in failed_screen_rows; no records were changed.')

,rows,screened_rows,missing_productbrand,missing_brand,failed_screen
_dataset,,,,,
Target,28082,27909,22,28082,173
Training,84552,84083,99,322,469


,outer_whitespace_changes,literal_None_kept
ProductBrand,2,1
Brand,3,0


Failed-screen records remain in failed_screen_rows; no records were changed.


## 2. Basic relationship strength

`row_count` includes missing Brand rows; `labeled_count` does not. Dominant share uses **populated labels only**, and exception count never counts missing labels. Ties have no unique expected Brand: the alphabetically first label is displayed only for reproducibility, and tied groups cannot become strong mappings.

“Deterministic” means one observed populated label, even if support is just one row. We report both coverage of all rows carrying that ProductBrand and coverage of populated-label rows. Those are different denominators.

In [3]:
def summarize(frame):
    known = frame.dropna(subset=['_ProductBrand'])
    result = known.groupby('_ProductBrand').agg(
        row_count=('_Brand', 'size'), labeled_count=('_Brand', 'count'),
        unique_brand_labels=('_Brand', 'nunique'),
    )
    pairs = (known.dropna(subset=['_Brand'])
             .groupby(['_ProductBrand', '_Brand']).size().rename('count').reset_index())
    pairs = pairs.sort_values(['_ProductBrand', 'count', '_Brand'], ascending=[True, False, True])
    pairs['is_top'] = pairs['count'].eq(pairs.groupby('_ProductBrand')['count'].transform('max'))
    leaders = pairs.drop_duplicates('_ProductBrand').set_index('_ProductBrand')
    result['dominant_brand'] = leaders['_Brand']
    result['dominant_count'] = leaders['count'].reindex(result.index).fillna(0).astype(int)
    result['top_tied'] = pairs.groupby('_ProductBrand')['is_top'].sum().reindex(result.index).fillna(0).gt(1)
    result['dominant_share'] = result['dominant_count'] / result['labeled_count'].replace(0, float('nan'))
    result['exception_count'] = result['labeled_count'] - result['dominant_count']
    result['missing_count'] = result['row_count'] - result['labeled_count']
    return result

raw_mapping = summarize(train_all)
mapping = summarize(train)

def overview(frame, table):
    deterministic = table['unique_brand_labels'].eq(1)
    observed = table['unique_brand_labels'].gt(0)
    return pd.Series({
        'rows': len(frame), 'unique_productbrands': len(table),
        'productbrands_with_labels': observed.sum(),
        'deterministic_productbrands': deterministic.sum(),
        'deterministic_pct_all_productbrands': 100 * deterministic.mean(),
        'deterministic_pct_with_labels': 100 * deterministic.sum() / observed.sum(),
        'all_row_coverage_pct': 100 * table.loc[deterministic, 'row_count'].sum() / len(frame),
        'labeled_deterministic_rows_pct_all_rows': 100 * table.loc[deterministic, 'labeled_count'].sum() / len(frame),
        'labeled_row_coverage_pct': 100 * table.loc[deterministic, 'labeled_count'].sum() / frame['_Brand'].notna().sum(),
        'deterministic_singletons': (deterministic & table['labeled_count'].eq(1)).sum(),
    })
relationship_overview = pd.DataFrame({
    'full_training': overview(train_all, raw_mapping),
    'screened_training': overview(train, mapping),
}).T
display(relationship_overview.round(3))
display(mapping.sort_values('row_count', ascending=False).head(25))
display(mapping.reindex(['Tylenol', 'Benadryl', 'BIOTICS', 'Preparation H']))

screen_sensitivity = raw_mapping.add_prefix('full_').join(mapping.add_prefix('screen_'))
screen_sensitivity['mode_changed'] = (
    screen_sensitivity['full_dominant_brand'].ne(screen_sensitivity['screen_dominant_brand']).fillna(False)
)
screen_sensitivity['share_change'] = screen_sensitivity['screen_dominant_share'] - screen_sensitivity['full_dominant_share']
print('ProductBrands absent after screening:', screen_sensitivity['screen_row_count'].isna().sum())
print('Dominant labels changed among comparable groups:', screen_sensitivity['mode_changed'].sum())
display(screen_sensitivity.loc[screen_sensitivity['mode_changed'],
    ['full_dominant_brand', 'screen_dominant_brand', 'full_labeled_count', 'screen_labeled_count', 'share_change']].head(20))
print('The earlier 15,565 / 14,868 counts are a comparison point, not imposed results.')
print('Here we explicitly distinguish all ProductBrands from those with populated labels,')
print('preserve case and literal None, trim outer whitespace, and show screening separately.')


,rows,unique_productbrands,productbrands_with_labels,deterministic_productbrands,deterministic_pct_all_productbrands,deterministic_pct_with_labels,all_row_coverage_pct,labeled_deterministic_rows_pct_all_rows,labeled_row_coverage_pct,deterministic_singletons
full_training,84552.0,15618.0,15568.0,14871.0,95.217,95.523,83.407,83.199,83.517,8248.0
screened_training,84083.0,15517.0,15467.0,14775.0,95.218,95.526,83.388,83.180,83.497,8192.0


,row_count,labeled_count,unique_brand_labels,dominant_brand,dominant_count,top_tied,dominant_share,exception_count,missing_count
_ProductBrand,,,,,,,,,
Nature Made,871,871,1,Nature Made,871,False,1.000000,0,0
NOW,774,774,1,NOW,774,False,1.000000,0,0
Walgreens,712,712,3,Walgreens,707,False,0.992978,5,0
CVS Health,693,693,3,CVS Health,604,False,0.871573,89,0
Nature's Bounty,690,690,1,Nature's Bounty,690,False,1.000000,0,0
Swanson,573,573,1,Swanson,573,False,1.000000,0,0
Generic,540,540,1,Generic,540,False,1.000000,0,0
Nature's Way,534,534,1,Nature's Way,534,False,1.000000,0,0
EQUATE,512,512,1,Equate,512,False,1.000000,0,0


,row_count,labeled_count,unique_brand_labels,dominant_brand,dominant_count,top_tied,dominant_share,exception_count,missing_count
_ProductBrand,,,,,,,,,
Tylenol,301,301,2,Tylenol,300,False,0.996678,1,0
Benadryl,124,124,2,Benadryl,123,False,0.991935,1,0
BIOTICS,104,104,2,UnItemised brand,103,False,0.990385,1,0
Preparation H,88,82,2,Other Brands,41,True,0.500000,41,6


ProductBrands absent after screening: 101
Dominant labels changed among comparable groups: 2


,full_dominant_brand,screen_dominant_brand,full_labeled_count,screen_labeled_count,share_change
_ProductBrand,,,,,
EverOne,UnItemised brand,Other Brands,3,2.0,-0.166667
Sonoma Lavender,Other Brands,UnItemised brand,2,1.0,0.500000


The earlier 15,565 / 14,868 counts are a comparison point, not imposed results.
Here we explicitly distinguish all ProductBrands from those with populated labels,
preserve case and literal None, trim outer whitespace, and show screening separately.


## 3. Look at the distribution before choosing a cutoff

The first table includes every mapping; the second isolates multi-label groups so the deterministic majority doesn't hide the interesting tail. Frequency tables show dominant share, exception count, and support. These describe the data; the bins are just display choices.

In [4]:
ambiguous = mapping.loc[mapping['unique_brand_labels'].gt(1)]
display(mapping[['labeled_count', 'dominant_share', 'exception_count']].describe(
    percentiles=[.01, .05, .10, .25, .5, .75, .9, .95, .99]).round(4))
display(ambiguous[['labeled_count', 'dominant_share', 'exception_count']].describe(
    percentiles=[.10, .25, .5, .75, .9, .95, .99]).round(4))

share_bins = pd.cut(ambiguous['dominant_share'], [0, .5, .75, .9, .95, .98, .99, 1], include_lowest=True)
share_distribution = ambiguous.groupby(share_bins, observed=False).agg(
    productbrands=('row_count', 'size'), labeled_rows=('labeled_count', 'sum'),
    exceptions=('exception_count', 'sum'),
)
display(share_distribution)
display(pd.cut(ambiguous['exception_count'], [0, 1, 2, 5, 10, 25, 100, float('inf')])
        .value_counts(sort=False).rename('productbrands').to_frame())
display(pd.cut(mapping['labeled_count'], [-1, 0, 1, 4, 9, 19, 49, 99, float('inf')])
        .value_counts(sort=False).rename('productbrands').to_frame())

,labeled_count,dominant_share,exception_count
count,15517.0000,15467.0000,15517.0000
mean,5.3920,0.9886,0.0971
std,24.3373,0.0634,1.0380
min,0.0000,0.3333,0.0000
1%,1.0000,0.5714,0.0000
5%,1.0000,1.0000,0.0000
10%,1.0000,1.0000,0.0000
25%,1.0000,1.0000,0.0000
50%,1.0000,1.0000,0.0000
75%,3.0000,1.0000,0.0000


,labeled_count,dominant_share,exception_count
count,692.0000,692.0000,692.0000
mean,19.8382,0.7453,2.1777
std,49.3625,0.1668,4.4333
min,2.0000,0.3333,1.0000
10%,2.0000,0.5000,1.0000
25%,3.0000,0.6000,1.0000
50%,7.0000,0.7500,1.0000
75%,18.0000,0.8889,2.0000
90%,48.9000,0.9599,4.0000
95%,79.9000,0.9808,6.0000


,productbrands,labeled_rows,exceptions
dominant_share,,,
"(-0.001, 0.5]",144,494,250
"(0.5, 0.75]",213,1554,527
"(0.75, 0.9]",177,3053,459
"(0.9, 0.95]",78,2279,166
"(0.95, 0.98]",44,1815,62
"(0.98, 0.99]",24,1850,26
"(0.99, 1.0]",12,2683,17


,productbrands
exception_count,
"(0.0, 1.0]",473
"(1.0, 2.0]",97
"(2.0, 5.0]",79
"(5.0, 10.0]",28
"(10.0, 25.0]",11
"(25.0, 100.0]",4
"(100.0, inf]",0


,productbrands
labeled_count,
"(-1.0, 0.0]",50
"(0.0, 1.0]",8192
"(1.0, 4.0]",4507
"(4.0, 9.0]",1416
"(9.0, 19.0]",717
"(19.0, 49.0]",407
"(49.0, 99.0]",125
"(99.0, inf]",103


### How sensitive is “strong” to the choice?

Compare several share and support cutoffs before choosing one. Support means count of the **dominant label**, not the total group size. Exceptions stay in the table even at high shares; we aren't assuming they're wrong. Candidate totals here exclude failed-screen rows and keep training and target separate.

In [5]:
sensitivity_rows = []
for share in [.90, .95, .98, .99, 1.0]:
    for support in [5, 10, 20, 50]:
        selected = mapping.loc[mapping['dominant_share'].ge(share)
                               & mapping['dominant_count'].ge(support) & ~mapping['top_tied']]
        sensitivity_rows.append({
            'min_share': share, 'min_dominant_support': support,
            'productbrands': len(selected), 'labeled_rows': selected['labeled_count'].sum(),
            'conflicts_training': selected['exception_count'].sum(),
            'missing_training': (train['_Brand'].isna() & train['_ProductBrand'].isin(selected.index)).sum(),
            'missing_target': (target['_Brand'].isna() & target['_ProductBrand'].isin(selected.index)).sum(),
        })
cutoff_sensitivity = pd.DataFrame(sensitivity_rows)
display(cutoff_sensitivity)

,min_share,min_dominant_support,productbrands,labeled_rows,conflicts_training,missing_training,missing_target
0,0.90,5,2507,59596,289,108,19164
1,0.90,10,1243,51425,283,76,16678
2,0.90,20,582,42604,228,52,13962
3,0.90,50,217,31710,143,26,10517
4,0.95,5,2423,57337,115,99,18416
5,0.95,10,1165,49226,115,67,15957
6,0.95,20,552,41093,112,44,13433
7,0.95,50,206,30803,74,18,10198
8,0.98,5,2374,55322,43,92,17689
9,0.98,10,1116,47211,43,60,15230


### Three observed cases, plus insufficient evidence

For this exploratory pass, call a multi-label mapping **near-deterministic at 99%**. This selects the extreme upper tail shown above and includes the supplied high-purity examples. It is a review cutoff, not a learned or validated cleaning rule. Separately require **20 dominant-label rows** for the “strong” candidate tables. The grid above shows the cost of changing either setting; 20 repeated listings still aren't 20 independent confirmations.

A deterministic singleton remains deterministic *in this sample* but isn't strong. “Genuinely ambiguous” below means **unresolved multi-label mapping below this cutoff or tied**, not proof that every competing label is correct. Groups with no populated labels have insufficient evidence and can't honestly fit any of the three mapping cases.

In [6]:
NEAR_SHARE = .99
MIN_SUPPORT = 20
mapping['mapping_case'] = 'insufficient evidence'
mapping.loc[mapping['unique_brand_labels'].eq(1), 'mapping_case'] = 'deterministic'
multi = mapping['unique_brand_labels'].gt(1)
mapping.loc[multi, 'mapping_case'] = 'genuinely ambiguous (unresolved)'
mapping.loc[multi & mapping['dominant_share'].ge(NEAR_SHARE) & ~mapping['top_tied'],
            'mapping_case'] = 'near-deterministic'
mapping['strong'] = (mapping['dominant_share'].ge(NEAR_SHARE)
                     & mapping['dominant_count'].ge(MIN_SUPPORT) & ~mapping['top_tied'])
strong_mapping = mapping.loc[mapping['strong']].copy()
record_cases = train.join(mapping[['mapping_case', 'strong']], on='_ProductBrand')
record_cases['mapping_case'] = record_cases['mapping_case'].fillna('missing ProductBrand')
display(mapping.groupby('mapping_case').agg(
    productbrands=('row_count', 'size'), rows=('row_count', 'sum'),
    labeled_rows=('labeled_count', 'sum'), missing_rows=('missing_count', 'sum'),
    strong_productbrands=('strong', 'sum'),
))
print('Record cases:', record_cases['mapping_case'].value_counts().to_dict())
display(mapping.reindex(['Tylenol', 'Benadryl', 'BIOTICS', 'Preparation H']))
print('Strong groups:', len(strong_mapping))

,productbrands,rows,labeled_rows,missing_rows,strong_productbrands
mapping_case,,,,,
deterministic,14775,70115,69940,175,470
genuinely ambiguous (unresolved),679,11033,10945,88,0
insufficient evidence,50,55,0,55,0
near-deterministic,13,2784,2783,1,13


Record cases: {'deterministic': 70115, 'genuinely ambiguous (unresolved)': 11033, 'near-deterministic': 2784, 'missing ProductBrand': 96, 'insufficient evidence': 55}


,row_count,labeled_count,unique_brand_labels,dominant_brand,dominant_count,top_tied,dominant_share,exception_count,missing_count,mapping_case,strong
_ProductBrand,,,,,,,,,,,
Tylenol,301,301,2,Tylenol,300,False,0.996678,1,0,near-deterministic,True
Benadryl,124,124,2,Benadryl,123,False,0.991935,1,0,near-deterministic,True
BIOTICS,104,104,2,UnItemised brand,103,False,0.990385,1,0,near-deterministic,True
Preparation H,88,82,2,Other Brands,41,True,0.500000,41,6,genuinely ambiguous (unresolved),False


Strong groups: 483


## 4. Missing Brand: proposals for inspection only

Apply the screened-training mapping to each dataset separately. Target labels never contribute support. `expected_brand` is the observed mode, not an approved replacement. Full tables retain original text, source record IDs, retailer, product name, and category. The previews show 20 rows per dataset.

In [7]:
evidence = strong_mapping[['dominant_brand', 'dominant_count', 'dominant_share', 'labeled_count',
                           'exception_count', 'mapping_case']].rename(columns={
    'dominant_brand': 'expected_brand', 'dominant_count': 'dominant_support',
})
scored = data.join(evidence, on='_ProductBrand')
strong_row = scored['expected_brand'].notna()
missing_all = scored.loc[strong_row & scored['_Brand'].isna()].copy()
missing_candidates = missing_all.loc[missing_all['_screen']].copy()
missing_screen_blocked = missing_all.loc[~missing_all['_screen']].copy()
review_columns = ['_dataset', '_source_row', 'ProductBrand', 'Brand', 'expected_brand',
                  'dominant_support', 'dominant_share', 'Retailer', 'ProductName', 'ProductCategory']
missing_summary = scored.groupby('_dataset').agg(
    total_rows=('_source_row', 'size'), missing_brand=('_Brand', lambda s: s.isna().sum()),
)
missing_summary['strong_mapping_candidates'] = missing_candidates.groupby('_dataset').size()
missing_summary['strong_but_failed_screen'] = missing_screen_blocked.groupby('_dataset').size()
display(missing_summary.fillna(0).astype(int))
missing_expected_counts = missing_candidates.groupby(['_dataset', 'expected_brand']).size().rename('candidate_rows')
display(missing_expected_counts.sort_values(ascending=False).head(25).to_frame())
for name in FILES:
    print(name, 'missing-label candidate preview')
    display(missing_candidates.loc[missing_candidates['_dataset'].eq(name), review_columns].head(20))

,total_rows,missing_brand,strong_mapping_candidates,strong_but_failed_screen
_dataset,,,,
Target,28082,28082,12165,26
Training,84552,322,33,0


candidate_rows
_dataset expected_brand                     
Target   UnItemised brand               4229
         NOW                             414
         Nature Made                     298
         Nature's Bounty                 285
         Swanson                         261
         Walgreens                       213
         Nature's Way                    196
         Generic                         191
         Equate                          168
         SOLARAY                         161
         Vicks                           147
         Nutricost                       145
         BulkSupplements                 141
         Life Extension                  133
         Puritan's Pride                 129
         Carlyle                         122
         Solgar                          117
         Source Naturals                 116
         Garden of Life                  116
         Tylenol                         100
         Mucinex                          97
         Pure Encapsulations              97
         GNC                              96
         Advil                            96
         Natrol                           94

Training missing-label candidate preview


,_dataset,_source_row,ProductBrand,Brand,expected_brand,dominant_support,dominant_share,Retailer,ProductName,ProductCategory
5276,Training,5276,Viva Naturals,null,UnItemised brand,27.0,1.0,Walmart,"Viva Naturals Organic Cacao Nibs, Whole Bean Coffee, 1 lb",Food>Baking>Baking Ingredients>Chocolate Chips & Cocoa
5287,Training,5287,Benefiber,null,UnItemised brand,52.0,1.0,Amazon,"Benefiber Drink Mix, Taste Free, 28 Stick Packs (Pack of 2)","Grocery & Gourmet Food > Beverages > Bottled Beverages, Water & Drink Mixes > Pow..."
5348,Training,5348,ICU Eyewear,null,UnItemised brand,29.0,1.0,Target,ICU Eyewear Classic Rectangular Reading Glasses +1.25 - 4pk,Target > Health > Eye Care
5374,Training,5374,Flora,null,UnItemised brand,44.0,1.0,Walmart,Flora - Flor Essence Herbal Tea Blend Liquid - 17 fl. oz.,Food>Beverages>Tea>Herbal Tea
5417,Training,5417,Little Remedies,null,UnItemised brand,36.0,1.0,Amazon,Little Remedies Saline Spray and Drops | Safe for Newborns | 0.5 Fl. Oz (Pack of 3),Health & Household > Health Care > OTC Medications & Treatments > Cough & Cold > ...
10917,Training,10917,Pink Stork,null,UnItemised brand,70.0,1.0,Amazon,"Pink Stork Total Nausea: Morning Sickness Relief for Pregnant Women, Nausea Relie...",Health & Household > Health Care > OTC Medications & Treatments > Digestion & Nau...
16169,Training,16169,Gaviscon,null,UnItemised brand,33.0,1.0,Target,Gaviscon Extra Strength Antacid - Original (100 Tablets),Target > Health > Medicines & Treatments > Digestion & Nausea
16206,Training,16206,Vital Proteins,null,Vital Proteins,127.0,1.0,Walmart,"Vital Proteins Vanilla Collagen Creamer, 10.6 oz, Protein Powder Supplement",Food>Coffee>Coffee Creamers>All Creamers
16484,Training,16484,Geri-Care,null,UnItemised brand,28.0,1.0,Amazon,"Senna Plus Natural Vegetable Laxative with Stool Softener, 100 tablets (Pack of 2)",Health & Household > Health Care > OTC Medications & Treatments > Digestion & Nau...
27175,Training,27175,ICU Eyewear,null,UnItemised brand,29.0,1.0,Target,ICU Eyewear Wink Rocklin Tortoise Floral Reading Glasses +1.25 - 2ct,Target > Health > Eye Care


Target missing-label candidate preview


,_dataset,_source_row,ProductBrand,Brand,expected_brand,dominant_support,dominant_share,Retailer,ProductName,ProductCategory
84552,Target,0,Swanson,null,Swanson,573.0,1.000000,Amazon,Swanson Prebiotic + Probiotic Fiber - Natural Supplement Promoting Digestive Syst...,"Health & Household > Vitamins, Minerals & Supplements > Digestive Supplements > D..."
84554,Target,2,MICHAEL'S,null,UnItemised brand,20.0,1.000000,Amazon,Michael's Naturopathic Programs Kidney Factors - 120 Vegetarian Tablets - Nutrien...,"Health & Household > Vitamins, Minerals & Supplements > Blended Vitamin & Mineral..."
84556,Target,4,Toplux,null,UnItemised brand,23.0,1.000000,Amazon,"Toplux Keto Pills Ketosis Diet - Natural Ketosis Using Ketone & Ketogenic Diet, S...","Health & Household > Vitamins, Minerals & Supplements > Essential Fatty Acids > 7..."
84558,Target,6,Om Mushroom Superfood,null,UnItemised brand,34.0,1.000000,Amazon,"Om Mushroom Superfood Brain Fuel Mushroom Powder Capsules Superfood Supplement, 9...","Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements > Mush..."
84560,Target,8,Ener-C,null,UnItemised brand,23.0,1.000000,Amazon,"Ener-C Tangerine Grapefruit Multivitamin Drink Mix, 1000mg Vitamin C, Non-GMO, Ve...","Health & Household > Vitamins, Minerals & Supplements > Vitamins > Vitamin C"
84562,Target,10,Benadryl,null,Benadryl,123.0,0.991935,Walmart,"Benadryl Original Strength Itch Stoping Cream - 1 Oz, 2 Pack",Health>Allergy and Sinus>Benadryl Anti Itch
84565,Target,13,NEURIVA,null,UnItemised brand,76.0,1.000000,Walmart,"Neuriva Plus Brain Performance Supplement (30 count), Brain Support With Clinical...",Health and Medicine > Vitamins and Supplements > Cognitive Health Supplements
84566,Target,14,Vitafusion,null,Vitafusion,196.0,1.000000,Kroger,vitafusion Melatonin Max Strength Dietary Supplement Gummies,Home > Health Care > Specialty Treatments
84569,Target,17,Lifeable,null,UnItemised brand,45.0,1.000000,Amazon,Lifeable Vitamin E 268mg - Great Tasting Natural Flavor Gummy Supplement - Vegeta...,"Health & Household > Vitamins, Minerals & Supplements > Vitamins > Vitamin E"
84573,Target,21,Viteey,null,UnItemised brand,68.0,1.000000,Amazon,"Probiotics Plus Prebiotics Fiber Gummies, Extra Strength 2 Billion CFUs for Immun...","Health & Household > Vitamins, Minerals & Supplements > Digestive Supplements > P..."


## 5. Populated conflicts: a separate review queue

These rows already have labels. Their disagreement with the mode is a reason to investigate, not permission to overwrite. No missing rows appear here. Support includes the other labeled rows of the group; this is an in-sample audit, not an accuracy estimate.

In [8]:
conflict_mask = strong_row & scored['_Brand'].notna() & scored['_Brand'].ne(scored['expected_brand']).fillna(False)
conflicts_all = scored.loc[conflict_mask].copy()
conflict_candidates = conflicts_all.loc[conflicts_all['_screen']].copy()
conflict_screen_blocked = conflicts_all.loc[~conflicts_all['_screen']].copy()
conflict_summary = pd.DataFrame({
    'conflict_candidates': conflict_candidates.groupby('_dataset').size(),
    'failed_screen_conflicts': conflict_screen_blocked.groupby('_dataset').size(),
}).reindex(FILES).fillna(0).astype(int)
display(conflict_summary)
display(conflict_candidates[review_columns].sort_values(['ProductBrand', '_dataset']).head(50))
conflict_pairs = conflict_candidates.groupby(['ProductBrand', 'Brand', 'expected_brand']).size().rename('rows')
display(conflict_pairs.sort_values(ascending=False).to_frame())

,conflict_candidates,failed_screen_conflicts
_dataset,,
Training,18,0
Target,0,0


,_dataset,_source_row,ProductBrand,Brand,expected_brand,dominant_support,dominant_share,Retailer,ProductName,ProductCategory
3612,Training,3612,ANDREW LESSMAN,UnItemised brand,Andrew Lessman,121.0,0.991803,Amazon,ANDREW LESSMAN Aloe Vera 400 - 60 Capsules – Provides 200:1 Ultra-Concentrate of ...,"Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements > Aloe..."
54181,Training,54181,Ark Labs,UnItemised brand,Ark Labs,112.0,0.991150,Amazon,Puradrop Extra Strength Advanced Formula Puradrops Gummies (2 Pack),"Health & Household > Health Care > OTC Medications & Treatments > Allergy, Sinus ..."
38029,Training,38029,BIOTICS,Other Brands,UnItemised brand,103.0,0.990385,Amazon,"BIOTICS Research Bio Allay Supports Healthy Inflammatory Responses, Joint Flexion...",Health & Household > Health Care > OTC Medications & Treatments > Pain Relievers ...
16983,Training,16983,Benadryl,Kirkland Signature,Benadryl,123.0,0.991935,Walmart,"Kirkland Signature Allergy Medicine 25 mg., 600 Minitabs",Health and Medicine > Cough Cold and Flu medicine > Cold Cough and Flu > Sore Thr...
32572,Training,32572,Force Factor,Other Brands,Force Factor,117.0,0.991525,Amazon,"Force Factor Tart Cherry Soft Chews for Joint Health, Exercise Recovery, Inflamma...",Health & Household > Health Care > OTC Medications & Treatments > Pain Relievers ...
37899,Training,37899,Icy Hot,Other Brands,Icy Hot,121.0,0.991803,Walmart,"Icy Hot Original Pain Relief Foam for Muscles and Joints, 4 oz",Health and Medicine>Pain management>Hot & Cold Therapy
59480,Training,59480,Mason Natural,UnItemised brand,Mason Natural,141.0,0.992958,Amazon,"Mason Natural, Glucosamine Chondroitin, 90 Capsules, Dietary Supplement Supports ...","Health & Household > Vitamins, Minerals & Supplements > Chondroitin & Glucosamine..."
44681,Training,44681,New Chapter,Other Brands,New Chapter,202.0,0.995074,Walgreens,Golden Black Seed,Shop>Vitamins & Supplements>Supplements>All Other Supplements
30864,Training,30864,ReNew Life,Other Brands,ReNew Life,139.0,0.992857,Amazon,"Renew Life Adult Probiotic - Ultimate Flora Probiotic Extra Care Go Pack, Shelf S...","Health & Household > Vitamins, Minerals & Supplements > Digestive Supplements > P..."
59503,Training,59503,SOLARAY,UnItemised brand,SOLARAY,480.0,0.995851,Walmart,"Solaray Fermented Lionâ€™s Mane Mushroom | Healthy Brain Function, Mental Focus &...",Health and Medicine > Superfoods & Cleanses > Superfood Caps


,,,rows
ProductBrand,Brand,expected_brand,
Walgreens,UnItemised brand,Walgreens,4
SOLARAY,UnItemised brand,SOLARAY,2
BIOTICS,Other Brands,UnItemised brand,1
Benadryl,Kirkland Signature,Benadryl,1
ANDREW LESSMAN,UnItemised brand,Andrew Lessman,1
Ark Labs,UnItemised brand,Ark Labs,1
Icy Hot,Other Brands,Icy Hot,1
Force Factor,Other Brands,Force Factor,1
New Chapter,Other Brands,New Chapter,1


## 6. What are the two taxonomy buckets doing?

Compare full and screened training counts first. Then inspect the screened view. “Named” here means any populated label outside these two buckets; it does **not** automatically mean that the label equals ProductBrand.

Count overlap in both directions: how many ProductBrands cross buckets, and how much of each bucket's row mass those groups represent. A long tail of rare ProductBrands can otherwise make overlap look larger or smaller than it is.

In [9]:
bucket_counts = []
for view_name, frame in [('full_training', train_all), ('screened_training', train)]:
    for label in BUCKETS:
        rows = frame.loc[frame['_Brand'].eq(label).fillna(False)]
        bucket_counts.append({'view': view_name, 'Brand': label, 'rows': len(rows),
                              'unique_productbrands': rows['_ProductBrand'].nunique(),
                              'missing_productbrand_rows': rows['_ProductBrand'].isna().sum()})
bucket_summary = pd.DataFrame(bucket_counts)
display(bucket_summary)
for label in BUCKETS:
    print(label, '— most common ProductBrands')
    display(train.loc[train['_Brand'].eq(label).fillna(False), '_ProductBrand']
            .value_counts().head(20).rename('rows').to_frame())

pair_counts = (train.dropna(subset=['_ProductBrand', '_Brand'])
               .groupby(['_ProductBrand', '_Brand']).size().rename('rows').reset_index())
bucket_wide = (pair_counts.loc[pair_counts['_Brand'].isin(BUCKETS)]
               .pivot(index='_ProductBrand', columns='_Brand', values='rows')
               .reindex(columns=BUCKETS).fillna(0).astype(int))
both_buckets = bucket_wide.loc[bucket_wide.gt(0).all(axis=1)].copy()
both_buckets['total_bucket_rows'] = both_buckets.sum(axis=1)
both_buckets['smaller_bucket_share'] = both_buckets[BUCKETS].min(axis=1) / both_buckets['total_bucket_rows']
print('ProductBrands appearing in either bucket:', len(bucket_wide))
print('ProductBrands appearing in BOTH buckets:', len(both_buckets))
for label in BUCKETS:
    print(label, 'row share from cross-bucket ProductBrands:',
          round(100 * both_buckets[label].sum() / bucket_wide[label].sum(), 2), '%')
display(both_buckets.sort_values('total_bucket_rows', ascending=False).head(30))

named_pairs = pair_counts.loc[~pair_counts['_Brand'].isin(BUCKETS)]
named_summary = named_pairs.groupby('_ProductBrand').agg(
    named_rows=('rows', 'sum'), named_labels=('_Brand', lambda s: ' | '.join(s)),
)
bucket_and_named = bucket_wide.join(named_summary, how='inner')
bucket_and_named['bucket_rows'] = bucket_and_named[BUCKETS].sum(axis=1)
bucket_and_named['named_share'] = bucket_and_named['named_rows'] / (bucket_and_named['named_rows'] + bucket_and_named['bucket_rows'])
exact_own = named_pairs.loc[named_pairs['_ProductBrand'].str.casefold().eq(named_pairs['_Brand'].str.casefold())]
bucket_and_named['own_name_rows'] = exact_own.groupby('_ProductBrand')['rows'].sum().reindex(bucket_and_named.index).fillna(0).astype(int)
print('ProductBrands with both bucket and nonbucket labels:', len(bucket_and_named))
print('Of those, with a case-insensitive exact own-name Brand:', bucket_and_named['own_name_rows'].gt(0).sum())
display(bucket_and_named.sort_values('bucket_rows', ascending=False).head(30))
# Keep a readable row-level table for later category / retailer inspection.
bucket_named_examples = train.loc[train['_ProductBrand'].isin(bucket_and_named.index),
    ['_source_row', 'ProductBrand', 'Brand', 'Retailer', 'ProductName', 'ProductCategory', 'Segment', 'Sub-Segment']]
example_brands = ['CVS Health', 'Walgreens', 'ZzzQuil', 'Preparation H']
example_rows = train.loc[train['_ProductBrand'].isin(example_brands)]
display(example_rows.groupby(['_ProductBrand', '_Brand'], dropna=False).head(2)[
    ['ProductBrand', 'Brand', 'Retailer', 'ProductName', 'ProductCategory']])
# A small context breakdown helps distinguish taxonomy structure from mere rarity.
bucket_context_mix = (train.loc[train['_Brand'].isin(BUCKETS)]
    .groupby(['_Brand', '_Retailer', '_Segment'], dropna=False).size()
    .rename('rows').reset_index().sort_values(['_Brand', 'rows'], ascending=[True, False]))
display(bucket_context_mix.groupby('_Brand', sort=False).head(8))

,view,Brand,rows,unique_productbrands,missing_productbrand_rows
0,full_training,UnItemised brand,54059,13695,93
1,full_training,Other Brands,3485,2255,2
2,screened_training,UnItemised brand,53640,13604,93
3,screened_training,Other Brands,3461,2241,2


UnItemised brand — most common ProductBrands


,rows
_ProductBrand,
Klaire Labs,104
BIOTICS,103
kivus,102
Dulcolax,102
S.O Labs,101
Systane,96
NutraChamps,94
Earthborn Elements,94
Terry Naturally,93


Other Brands — most common ProductBrands


,rows
_ProductBrand,
Preparation H,41
Dramamine,36
BREATHE RIGHT,27
Three Lollies,18
Ace,16
Sea-Band,16
MUELLER,16
Bonine,14
Tiger Balm,13


...

ProductBrands with both bucket and nonbucket labels: 65


Of those, with a case-insensitive exact own-name Brand: 38


,UnItemised brand,Other Brands,named_rows,named_labels,bucket_rows,named_share,own_name_rows
_ProductBrand,,,,,,,
CVS Health,88,1,604,CVS Health,89,0.871573,604
Major Pharmaceuticals,81,3,3,Sudafed,84,0.034483,0
ZzzQuil,75,0,3,Vicks,75,0.038462,0
HealthA2Z,38,3,1,Tylenol,41,0.023810,0
Unknown,23,0,3,GNC,23,0.115385,0
Amazon.com,22,1,4,Herb Pharm | Nature's Bounty,23,0.148148,0
GeriCare,20,2,1,Benadryl,22,0.043478,0
Pepto,19,1,31,Pepto Bismol,20,0.607843,0
Voltaren,7,10,7,Voltaren,17,0.291667,7


,ProductBrand,Brand,Retailer,ProductName,ProductCategory
6,CVS Health,UnItemised brand,CVS,CVS Health Hot/Cold Gel Slippers,Health & Medicine > Pain & Fever > Hot or Cold Therapy
84,CVS Health,CVS Health,CVS,CVS Health Dairy Relief Fast Acting Caplets,Health & Medicine > Digestive Health > Lactose Intolerance
114,CVS Health,CVS Health,CVS,CVS Health Ibuprofen 200 MG Liquid Filled Mini Softgels,Health & Medicine > Pain & Fever > Pain Relievers
121,Walgreens,Walgreens,Walgreens,Chewable Papaya Enzyme Complex Tablets Papaya,Shop>Vitamins & Supplements>Shop by Health Benefit>Digestive Support Supplements
468,ZzzQuil,UnItemised brand,Amazon,"ZzzQuil, Nighttime Sleep Aid Liquid, FREE OF Alcohol & Artificial Dyes, 50 mg Dip...",Health & Household > Health Care > OTC Medications & Treatments > Sleep Aids
555,Walgreens,Walgreens,Walgreens,Gas Relief Softgels,Shop>Medicines & Treatments>Digestive Health & Nausea>Gas Relief
1651,ZzzQuil,UnItemised brand,Amazon,"ZzzQuil Vicks Pure Zzzs Sleep + De-Stress Melatonin Gummies, 26 CT (Pack of 2)","Health & Household > Vitamins, Minerals & Supplements > Sleep Supplements"
2451,CVS Health,UnItemised brand,CVS,"CVS Health Cognitive Gummies, 90 CT",Vitamins > Supplements > Brain and Memory Support
5219,Preparation H,Other Brands,Amazon,Flushable Medicated Hemorrhoid Wipes,Health & Household > Health Care > OTC Medications & Treatments > Pain Relievers ...
5279,Preparation H,UnItemised brand,CVS,PREPARATION H Maximum Strength Multi-Symptom Hemorrhoid Cream,Health & Medicine > Pain & Fever > Hemorrhoid Treatment


,_Brand,_Retailer,_Segment,rows
8,Other Brands,Amazon,<NA>,718
34,Other Brands,Walmart,<NA>,701
2,Other Brands,Amazon,Digestive Health,618
4,Other Brands,Amazon,Internal Analgesics,610
7,Other Brands,Amazon,"Vitamins, Minerals & Supplements",239
0,Other Brands,Amazon,Allergy,218
25,Other Brands,Walgreens,<NA>,58
30,Other Brands,Walmart,Internal Analgesics,57
42,UnItemised brand,Amazon,"Vitamins, Minerals & Supplements",28212
38,UnItemised brand,Amazon,External Analgesics,4933


## 7. Does context change the relationship?

For every strong ProductBrand, compare the global mode with each Retailer, ProductCategory, Segment, and Sub-Segment group. These dimensions are checked separately, not as a joint rule. Missing contexts remain visible.

Show **every local disagreement**, including one-row contexts. A second count asks whether a different, untied local mode has at least five labeled rows. Five is an inspection aid, not proof of validity; small groups remain unresolved, not invalid. A single observed context also cannot establish stability across contexts. Segment and Sub-Segment may be missing at prediction time, so don't use them as assumed inputs for filling target labels.

In [10]:
strong_labeled = train.loc[train['_Brand'].notna() & train['_ProductBrand'].isin(strong_mapping.index)].copy()
context_tables = {}
context_summary_rows = []
for context in CONTEXTS:
    # Only the derived context key gets a missing-value display marker.
    local = strong_labeled.copy()
    local['_context'] = local['_' + context].fillna('<missing>')
    counts = local.groupby(['_ProductBrand', '_context', '_Brand']).size().rename('label_count').reset_index()
    counts = counts.sort_values(['_ProductBrand', '_context', 'label_count', '_Brand'],
                                ascending=[True, True, False, True])
    keys = ['_ProductBrand', '_context']
    counts['top'] = counts['label_count'].eq(counts.groupby(keys)['label_count'].transform('max'))
    leaders = counts.drop_duplicates(keys).set_index(keys)
    stats = counts.groupby(keys).agg(context_rows=('label_count', 'sum'),
                                    unique_labels=('_Brand', 'size'), top_tied=('top', 'sum'))
    stats['local_brand'] = leaders['_Brand']
    stats['local_support'] = leaders['label_count']
    stats['local_share'] = stats['local_support'] / stats['context_rows']
    stats = stats.reset_index().join(evidence[['expected_brand', 'dominant_share']], on='_ProductBrand')
    expected_counts = counts.join(evidence[['expected_brand']], on='_ProductBrand')
    expected_counts = expected_counts.loc[expected_counts['_Brand'].eq(expected_counts['expected_brand'])]
    expected_lookup = expected_counts.set_index(keys)['label_count']
    stats['global_label_count'] = stats.set_index(keys).index.map(expected_lookup).fillna(0).astype(int)
    stats['global_label_share'] = stats['global_label_count'] / stats['context_rows']
    stats['disagreement_rows'] = stats['context_rows'] - stats['global_label_count']
    stats['mode_differs'] = stats['local_brand'].ne(stats['expected_brand']) | stats['top_tied'].gt(1)
    stats['supported_alternative'] = (stats['mode_differs'] & stats['top_tied'].eq(1)
                                      & stats['context_rows'].ge(5))
    context_tables[context] = stats
    breadth = stats.loc[stats['_context'].ne('<missing>')].groupby('_ProductBrand').size()
    context_summary_rows.append({
        'context': context, 'groups': len(stats),
        'brands_with_2plus_observed_contexts': breadth.ge(2).sum(),
        'groups_with_any_disagreement': stats['disagreement_rows'].gt(0).sum(),
        'different_or_tied_local_modes': stats['mode_differs'].sum(),
        'supported_alternative_groups': stats['supported_alternative'].sum(),
        'missing_context_rows': stats.loc[stats['_context'].eq('<missing>'), 'context_rows'].sum(),
    })
context_summary = pd.DataFrame(context_summary_rows).set_index('context')
display(context_summary)
context_exceptions = pd.concat({c: t.loc[t['disagreement_rows'].gt(0)]
                                for c, t in context_tables.items()}, names=['dimension', 'table_row'])
display(context_exceptions.sort_values(['supported_alternative', 'context_rows'], ascending=False).head(40))
# Full detail: context_tables['Retailer'], context_tables['ProductCategory'], etc.
context_unstable_brands = set(context_exceptions['_ProductBrand'])
print('Strong ProductBrands with ANY context disagreement:', len(context_unstable_brands))
print('All contexts with a different or tied local mode (including low support):')
display(context_exceptions.loc[context_exceptions['mode_differs'],
    ['_ProductBrand', '_context', 'context_rows', 'local_brand', 'local_share',
     'expected_brand', 'global_label_share', 'top_tied']])


,groups,brands_with_2plus_observed_contexts,groups_with_any_disagreement,different_or_tied_local_modes,supported_alternative_groups,missing_context_rows
context,,,,,,
Retailer,1207,312,14,0,0,0
ProductCategory,12523,480,16,9,0,7
Segment,1632,409,14,3,0,1052
Sub-Segment,3842,470,14,2,0,2160


_ProductBrand  \
dimension       table_row                   
Retailer        1132            Walgreens   
                926               SOLARAY   
Segment         1225              SOLARAY   
                1427              Tylenol   
                964           New Chapter   
Sub-Segment     3349              Tylenol   
                3540            Walgreens   
Segment         148              Benadryl   
Retailer        14         ANDREW LESSMAN   
Segment         1512            Walgreens   
Retailer        88               Ark Labs   
Segment         791         Mason Natural   
Retailer        1066              Tylenol   
Segment         28         ANDREW LESSMAN   
Retailer        100               BIOTICS   
Sub-Segment     364              Benadryl   
Retailer        587         Mason Natural   
Segment         1193           ReNew Life   
Retailer        359          Force Factor   
Sub-Segment     2894              SOLARAY   
Retailer        928               SOLARAY   
                894            ReNew Life   
                125              Benadryl   
                727           New Chapter   
Sub-Segment     1879        Mason Natural   
Retailer        1186               Zyrtec   
Segment         627               Icy Hot   
Sub-Segment     2889              SOLARAY   
ProductCategory 11853           Walgreens   
                9467           ReNew Life   
Sub-Segment     1469              Icy Hot   
                63         ANDREW LESSMAN   
                2284          New Chapter   
Retailer        483               Icy Hot   
Segment         1220              SOLARAY   
Sub-Segment     251              Ark Labs   
ProductCategory 11867           Walgreens   
                11211             Tylenol   
Segment         440          Force Factor   
Sub-Segment     2826           ReNew Life   

                                                                                                       _context  \
dimension       table_row                                                                                         
Retailer        1132                                                                                  Walgreens   
                926                                                                                      Amazon   
Segment         1225                                                           Vitamins, Minerals & Supplements   
                1427                                                                        Internal Analgesics   
                964                                                            Vitamins, Minerals & Supplements   
Sub-Segment     3349                                                                              Internal Pain   
                3540                                                                                  <missing>   
Segment         148                                                                                     Allergy   
Retailer        14                                                                                       Amazon   
Segment         1512                                                                                  <missing>   
Retailer        88                                                                                       Amazon   
Segment         791                                                            Vitamins, Minerals & Supplements   
Retailer        1066                                                                                     Amazon   
Segment         28                                                             Vitamins, Minerals & Supplements   
Retailer        100                                                                                      Amazon   
Sub-Segment     364                                                                          Oral Antihistamine   
Retailer        587                                                

Strong ProductBrands with ANY context disagreement: 13
All contexts with a different or tied local mode (including low support):


_ProductBrand  \
dimension       table_row                   
ProductCategory 175        ANDREW LESSMAN   
                688              Ark Labs   
                793               BIOTICS   
                995              Benadryl   
                3247         Force Factor   
                4517              Icy Hot   
                9706              SOLARAY   
                11855           Walgreens   
                12346              Zyrtec   
Segment         98               Ark Labs   
                119               BIOTICS   
                1601               Zyrtec   
Sub-Segment     303               BIOTICS   
                1050         Force Factor   

                                                                                                       _context  \
dimension       table_row                                                                                         
ProductCategory 175        Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements > Aloe...   
                688        Health & Household > Health Care > OTC Medications & Treatments > Allergy, Sinus ...   
                793        Health & Household > Health Care > OTC Medications & Treatments > Pain Relievers ...   
                995        Health and Medicine > Cough Cold and Flu medicine > Cold Cough and Flu > Sore Thr...   
                3247       Health & Household > Health Care > OTC Medications & Treatments > Pain Relievers ...   
                4517                                     Health and Medicine>Pain management>Hot & Cold Therapy   
                9706                               Health and Medicine > Superfoods & Cleanses > Superfood Caps   
                11855                 Shop>Medicines & Treatments>Pain Relief & Management>Migraine Pain Relief   
                12346      Health and Medicine > Home Health Care > Bath Safety > Sitz Baths, Bedpans and Ur...   
Segment         98                                                                                      Allergy   
                119                                                                         Internal Analgesics   
                1601                                                                                  <missing>   
Sub-Segment     303                                                                             Speciality Pain   
                1050                                                                            Speciality Pain   

                           context_rows       local_brand  local_share  \
dimension       table_row                                                
ProductCategory 175                   1  UnItemised brand          1.0   
                688                   1  UnItemised brand          1.0   
                793                   1      Other Brands          1.0   
                995                   2          Benadryl          0.5   
                3247                  1      Other Brands          1.0   
                4517                  1      Other Brands          1.0   
                9706                  2           SOLARAY          0.5   
                11855                 2      Other Brands          0.5   
                12346                 1  UnItemised brand          1.0   
Segment         98                    1  UnItemised brand          1.0   
                119                   1      Other Brands          1.0   
                1601                  1  UnItemised brand          1.0   
Sub-Segment     303                   1      Other Brands          1.0   
                1050                  1      Other Brands          1.0   

                             expected_brand  global_label_share  top_tied  
dimension       table_row                                                  
ProductCategory 175          Andrew Lessman                 0.0         1  
                688                Ark Labs          

### Add context cautions to the review tables

For each candidate, show how many training rows support that exact context and whether any of them disagree with the global Brand. Unseen contexts, absent context values, and known disagreements all deserve caution. A globally deterministic brand in a new retailer/category is still an extrapolation.

These annotations narrow the discussion; they are not a cleaner or an authorization to fill.

In [11]:
def annotate_context(frame):
    result = frame.copy()
    for context, table in context_tables.items():
        lookup = table.set_index(['_ProductBrand', '_context'])
        keys = pd.MultiIndex.from_arrays([result['_ProductBrand'], result['_' + context].fillna('<missing>')])
        result[context + '_support'] = keys.map(lookup['context_rows']).fillna(0).astype(int)
        result[context + '_conflicts'] = keys.map(lookup['disagreement_rows']).fillna(0).astype(int)
        result[context + '_unseen_or_missing'] = result['_' + context].isna() | result[context + '_support'].eq(0)
    result['any_context_caution'] = (
        result[[c + '_unseen_or_missing' for c in CONTEXTS]].any(axis=1)
        | result[[c + '_conflicts' for c in CONTEXTS]].gt(0).any(axis=1)
    )
    result['globally_deterministic'] = result['exception_count'].eq(0)
    result['expected_is_bucket'] = result['expected_brand'].isin(BUCKETS)
    return result

missing_candidates = annotate_context(missing_candidates)
conflict_candidates = annotate_context(conflict_candidates)
candidate_cautions = missing_candidates.groupby('_dataset').agg(
    missing_candidates=('_source_row', 'size'),
    deterministic_mapping_rows=('globally_deterministic', 'sum'),
    expected_bucket_rows=('expected_is_bucket', 'sum'),
    rows_with_context_caution=('any_context_caution', 'sum'),
)
display(candidate_cautions)
display(conflict_candidates[review_columns + ['any_context_caution', 'expected_is_bucket']].head(30))
# Missing Segment/Sub-Segment should not hide what the observable product contexts say.
observable_caution = (missing_candidates['Retailer_unseen_or_missing']
                     | missing_candidates['ProductCategory_unseen_or_missing']
                     | missing_candidates['Retailer_conflicts'].gt(0)
                     | missing_candidates['ProductCategory_conflicts'].gt(0))
print('Missing candidates with retailer/category caution:', int(observable_caution.sum()))

,missing_candidates,deterministic_mapping_rows,expected_bucket_rows,rows_with_context_caution
_dataset,,,,
Target,12165,11282,4229,12165
Training,33,32,31,33


,_dataset,_source_row,ProductBrand,Brand,expected_brand,dominant_support,dominant_share,Retailer,ProductName,ProductCategory,any_context_caution,expected_is_bucket
3612,Training,3612,ANDREW LESSMAN,UnItemised brand,Andrew Lessman,121.0,0.991803,Amazon,ANDREW LESSMAN Aloe Vera 400 - 60 Capsules – Provides 200:1 Ultra-Concentrate of ...,"Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements > Aloe...",True,False
10290,Training,10290,Tylenol,Kirkland Signature,Tylenol,300.0,0.996678,Amazon,"Kirkland Signature Extra Strength WIpcLl Rapid Release Acetaminophen PM 500mg, 4 ...",Health & Household > Health Care > Over-the-Counter Medication > Pain Relievers >...,True,False
16983,Training,16983,Benadryl,Kirkland Signature,Benadryl,123.0,0.991935,Walmart,"Kirkland Signature Allergy Medicine 25 mg., 600 Minitabs",Health and Medicine > Cough Cold and Flu medicine > Cold Cough and Flu > Sore Thr...,True,False
27102,Training,27102,Walgreens,UnItemised brand,Walgreens,707.0,0.992978,Walgreens,Emu Muscle & Joint Relief,Shop>Medicines & Treatments>Pain Relief & Management>Joint & Muscle Pain Relief,True,False
27286,Training,27286,Walgreens,Other Brands,Walgreens,707.0,0.992978,Walgreens,Extra Strength Headache Relief Caplets,Shop>Medicines & Treatments>Pain Relief & Management>Migraine Pain Relief,True,False
30864,Training,30864,ReNew Life,Other Brands,ReNew Life,139.0,0.992857,Amazon,"Renew Life Adult Probiotic - Ultimate Flora Probiotic Extra Care Go Pack, Shelf S...","Health & Household > Vitamins, Minerals & Supplements > Digestive Supplements > P...",True,False
32572,Training,32572,Force Factor,Other Brands,Force Factor,117.0,0.991525,Amazon,"Force Factor Tart Cherry Soft Chews for Joint Health, Exercise Recovery, Inflamma...",Health & Household > Health Care > OTC Medications & Treatments > Pain Relievers ...,True,False
37899,Training,37899,Icy Hot,Other Brands,Icy Hot,121.0,0.991803,Walmart,"Icy Hot Original Pain Relief Foam for Muscles and Joints, 4 oz",Health and Medicine>Pain management>Hot & Cold Therapy,True,False
38029,Training,38029,BIOTICS,Other Brands,UnItemised brand,103.0,0.990385,Amazon,"BIOTICS Research Bio Allay Supports Healthy Inflammatory Responses, Joint Flexion...",Health & Household > Health Care > OTC Medications & Treatments > Pain Relievers ...,True,True
43433,Training,43433,Walgreens,UnItemised brand,Walgreens,707.0,0.992978,Walgreens,Maimum Strength Lidocaine Pain Patches Assorted Sizes,Shop>Medicines & Treatments>Pain Relief & Management>Joint & Muscle Pain Relief,True,False


Missing candidates with retailer/category caution: 2822


## 8. Evidence boundaries and checks

This is descriptive, in-sample evidence. There is no held-out accuracy estimate or manual ground-truth check here. Repeated listings, retailer-specific conventions, unobserved contexts, and possible source corruption can inflate apparent certainty. The provisional structural screen is also an assumption, so keep its comparison visible.

The full tables are `mapping`, `missing_candidates`, `conflict_candidates`, `bucket_and_named`, `both_buckets`, and `context_tables`. Failed-screen matches are separate in `missing_screen_blocked` and `conflict_screen_blocked`. Nothing is exported or applied.

In [12]:
assert data[['ProductBrand', 'Brand']].equals(original_labels)
assert missing_candidates['_Brand'].isna().all()
assert conflict_candidates['_Brand'].notna().all()
assert conflict_candidates['_Brand'].ne(conflict_candidates['expected_brand']).all()
assert set(missing_candidates.index).isdisjoint(conflict_candidates.index)
assert mapping['exception_count'].eq(mapping['labeled_count'] - mapping['dominant_count']).all()
assert mapping['row_count'].eq(mapping['labeled_count'] + mapping['missing_count']).all()
assert missing_candidates['_screen'].all() and conflict_candidates['_screen'].all()
assert {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in FILES.items()} == source_hashes
print('Checks passed: labels and source files unchanged; missing and conflicting queues are separate.')

# Compact evidence for the conclusion. Re-run all cells if data or thresholds change.
display(relationship_overview.round(3))
display(candidate_cautions)
display(conflict_summary)
display(context_summary)

Checks passed: labels and source files unchanged; missing and conflicting queues are separate.


,rows,unique_productbrands,productbrands_with_labels,deterministic_productbrands,deterministic_pct_all_productbrands,deterministic_pct_with_labels,all_row_coverage_pct,labeled_deterministic_rows_pct_all_rows,labeled_row_coverage_pct,deterministic_singletons
full_training,84552.0,15618.0,15568.0,14871.0,95.217,95.523,83.407,83.199,83.517,8248.0
screened_training,84083.0,15517.0,15467.0,14775.0,95.218,95.526,83.388,83.180,83.497,8192.0


,missing_candidates,deterministic_mapping_rows,expected_bucket_rows,rows_with_context_caution
_dataset,,,,
Target,12165,11282,4229,12165
Training,33,32,31,33


,conflict_candidates,failed_screen_conflicts
_dataset,,
Training,18,0
Target,0,0


,groups,brands_with_2plus_observed_contexts,groups_with_any_disagreement,different_or_tied_local_modes,supported_alternative_groups,missing_context_rows
context,,,,,,
Retailer,1207,312,14,0,0,0
ProductCategory,12523,480,16,9,0,7
Segment,1632,409,14,3,0,1052
Sub-Segment,3842,470,14,2,0,2160


## 9. What does this mean for cleaning?

**Observed evidence (this run):** In the full training export, 14,871 of 15,568 ProductBrands with populated labels are deterministic (**95.52%**). Including 50 ProductBrands with only missing labels gives 15,618 distinct keys. Deterministic keys cover **83.41% of all training rows**, including missing labels; their populated rows alone cover **83.20%**. These denominators and the explicit key handling explain why we report fresh counts rather than repeat the starting figures.

The screened view is similar, but **8,192 deterministic mappings have only one labeled row**. At 99% dominant share and 20 dominant-label rows, 483 mappings qualify: 470 deterministic and 13 near-deterministic. They produce **33 missing training candidates**, **12,165 missing target candidates**, and **18 populated training conflicts**. Another 26 missing target matches fail the structural screen and stay outside the queue. These are review counts, not validated corrections.

Retailer modes agree with the global mode in the observed strong groups. ProductCategory, Segment, and Sub-Segment have **9, 3, and 2 different-or-tied local modes**, respectively; none meets the five-row untied-alternative check. Small contexts remain unresolved. Only 312 of 483 strong ProductBrands occur across at least two observed retailers, so global retailer stability is not established for all groups. **2,822 missing candidates have unseen, missing, or conflicting retailer/category context**; all missing candidates have some caution when the two target-label contexts are included.

The screened training data has **53,640 `UnItemised brand` rows** across 13,604 known ProductBrands and **3,461 `Other Brands` rows** across 2,241. **640 ProductBrands appear in both buckets**, accounting for 12.97% and 39.17% of their respective rows. Another view finds 65 ProductBrands with both bucket and nonbucket labels; 38 have an exact own-name label ignoring case. Preparation H remains tied 41/41 among populated labels, with six missing labels.

**Interpretation:** The buckets' broad, repeated use is consistent with intentional taxonomy grouping. The overlap and named-versus-bucket switches leave assignment rules unresolved; counts cannot tell us which individual labels are inconsistent or what the Product Owner intended.

- **Hard rule?** No. Multi-label mappings, thin support, and unresolved context differences rule out a universal ProductBrand → Brand rule.
- **Safe missing fills?** Not established yet. Well-supported deterministic mappings are a useful validation shortlist. Confirm taxonomy rules and check independent products / held-out data before approving fills; near-deterministic groups and new contexts need extra review.
- **Conflict flags?** Yes, as a review signal. The 18 conflicts deserve inspection of product evidence and context. Being the minority is not evidence of invalidity.
- **Keep the buckets?** Yes. Preserve both as valid taxonomy values pending a confirmed rule; don't replace them simply because a named Brand exists elsewhere.
- **Leave untouched?** All labels in this notebook. In particular: ambiguous or tied groups, low support, missing/unseen ProductBrands, failed-screen rows, context disagreements, unseen contexts, and unresolved bucket-versus-named cases.

These conclusions describe the current files and settings. If either changes, re-run the notebook and revisit this interpretation.